In [1]:
# import numpy as np
# import pandas as pd 
# import matplotlib.pyplot as plt 
# import scipy.special as spsp
# import scipy.stats as spst

# from statsmodels.api import Logit
# from statsmodels.formula.api import poisson, negativebinomial, ols, logit
# from statsmodels.base.model import GenericLikelihoodModel

# import warnings
# warnings.filterwarnings('ignore')

In [9]:
%run "BlackjackSetup.ipynb"

In [10]:
# K arms
# d features

import numpy as np

def compute_ucb(x, A, b, alpha):
    UCB = []
    for i in range(len(b)):
        Ainv = np.linalg.inv(A[i])
        Rhat = x @ Ainv @ b[i]
        Uhat = alpha * np.sqrt(x.T @ Ainv @ x)
        UCB.append(Rhat+Uhat)
    return np.array(UCB)

tenv = BlackjackTable()

def linucb(alpha, trials):
    K = 2 # hit vs stay
    d = 39 # features
    A = [np.eye(d+1) for i in range(K)]
    b = [np.zeros(d+1) for i in range(K)]

    for n in range(trials):
        tenv.reset()
        d = False

        while not d:
            if tenv._get_obs()['bet'] == 0:
                tenv.step((0,1)) # initial bet
            
            player = interprethand(tenv._get_obs()['playercards'])
            # display(player)
            dealer = interprethand(tenv._get_obs()['dealercards'])
            prev_cards = interprethand(tenv._get_info()['pastcards'])
            x = np.concatenate((np.array([1]), prev_cards, player, dealer))

            UCB = compute_ucb(x, A, b, alpha)
            idx = np.argmax(UCB)

            step = tenv.step((1, idx)) # second param is dummy for bet
            rewards = step[1]
            d = step[2]
            # readoutput(step)
            
            A[idx] += np.outer(x,x)
            b[idx] += rewards*x
    
    return A, b

In [11]:
results = linucb(0.1, 100)
A, b = results


In [12]:
betas = np.zeros((len(b), len(b[0])))

for i in range(len(b)):
    Ainv = np.linalg.inv(A[i])
    betas[i] = Ainv @ b[i]

In [13]:
opt_action = np.zeros((13,13), dtype=int)

for i in range(13):
    for j in range(13):
        prev = np.zeros(13)
        player = np.zeros(13)
        dealer = np.zeros(13)

        player[i] = 1
        dealer[j] = 1

        x = np.concatenate((np.array([1]), prev, player, dealer))

        reward = betas @ x
        opt_action[i,j] = np.argmax(reward)

In [14]:
opt_action

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1],
       [1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])